In [ ]:
def main(datasources, start_date, end_date):
    """
    使用 DAI UDF 计算日内订单簿压力因子。

    评测时平台会注入 datasources、start_date 和 end_date。
    返回包含 ['date', 'instrument', 'factor'] 三列的 DataFrame。
    """
    import math
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    def calculate_weighted_imbalance(
        bid_volume1: float,
        bid_volume2: float,
        bid_volume3: float,
        bid_volume4: float,
        bid_volume5: float,
        ask_volume1: float,
        ask_volume2: float,
        ask_volume3: float,
        ask_volume4: float,
        ask_volume5: float,
    ) -> float:
        """计算与原 SQL 一致的五档加权订单簿不平衡度。"""
        weights = (1.0, math.exp(-0.3), math.exp(-0.6), math.exp(-0.9), math.exp(-1.2))
        bid_volumes = (bid_volume1, bid_volume2, bid_volume3, bid_volume4, bid_volume5)
        ask_volumes = (ask_volume1, ask_volume2, ask_volume3, ask_volume4, ask_volume5)
        weight_bid = sum(volume * weight for volume, weight in zip(bid_volumes, weights))
        weight_ask = sum(volume * weight for volume, weight in zip(ask_volumes, weights))
        return float((weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8))

    def label_time_segment(value) -> int:
        """按原 SQL 的左开右闭规则返回固定的 30 分钟截面。"""
        hhmmss = value.hour * 10000 + value.minute * 100 + value.second
        sections = (
            (90000, 93000),
            (93000, 100000),
            (100000, 103000),
            (103000, 110000),
            (110000, 113000),
            (130000, 133000),
            (133000, 140000),
            (140000, 143000),
        )
        for section_start, section_end in sections:
            if section_start < hhmmss <= section_end:
                return section_end
        return -1

    sql = f"""
    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            calculate_weighted_imbalance(
                COALESCE(bid_volume1, 0),
                COALESCE(bid_volume2, 0),
                COALESCE(bid_volume3, 0),
                COALESCE(bid_volume4, 0),
                COALESCE(bid_volume5, 0),
                COALESCE(ask_volume1, 0),
                COALESCE(ask_volume2, 0),
                COALESCE(ask_volume3, 0),
                COALESCE(ask_volume4, 0),
                COALESCE(ask_volume5, 0)
            ) AS weighted_imbalance,
            label_time_segment(date) AS time_segment
        FROM {bar1m}
        WHERE time_segment != -1
    ),
    cte_window AS (
        SELECT
            trading_day,
            time_segment,
            instrument_id,
            AVG(weighted_imbalance) AS factor
        FROM cte_snapshot
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.factor
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
        udf_list=[
            dai.DaiUDF(
                name="calculate_weighted_imbalance",
                function=calculate_weighted_imbalance,
            ),
            dai.DaiUDF(
                name="label_time_segment",
                function=label_time_segment,
            ),
        ],
    ).df()

    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["factor"])

    # 与原 SQL 版本一致，对齐中证 1000 股票池。
    stk_pool = dai.query(
        "SELECT date, instrument FROM cpt_jyc_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    result = pd.merge(df, stk_pool, how="inner", on=["date", "instrument"])

    return result[["date", "instrument", "factor"]]


if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()
    datasources = {"bar1m": "cpt_jyc_2026_stock_bar1m"}
    start_date = "2020-01-01 00:00:00"
    end_date = "2020-03-01 23:59:59"

    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    result = M.jyc_eval._latest(factor_data=factor_data, show=True)
